In [ ]:
from pathlib import Path

# os.chdir('..')
Path.cwd()

PosixPath('/home/emmanuel/Desktop/research_peft')

### Implementing the tokenizer

Things to do:
- load the tokenizer    (Done)
- load model            (Done)
- Download dataset 
- Preprocess dataset
- Tokenize dataset
- Create DataLoader
- Configure Trainer
- Train model
- Save checkpoint
- Load checkpoint
- Evaluate checkpoint

In [ ]:
from dataclasses import dataclass


@dataclass
class ModelConfig:
    model_name_or_path: str = "Qwen/Qwen2.5-1.5B"
    tokenizer_name: str = None
    trust_remote_code: bool = True

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


# loading the tokenizer
def load_tokenizer(model_cfg: ModelConfig):
    name = model_cfg.tokenizer_name or model_cfg.model_name_or_path
    tokenizer = AutoTokenizer.from_pretrained(
        name,
        trust_remote_code=model_cfg.trust_remote_code,
        padding_side="right",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

In [ ]:
from typing import Any

import torch


def load_base_model(model_cfg: ModelConfig):
    """Load base model"""

    model_kwargs: dict[str, Any] = {
        "pretrained_model_name_or_path": model_cfg.model_name_or_path,
        "trust_remote_code": model_cfg.trust_remote_code,
        "torch_dtype": torch.bfloat16,
        "device_map": "auto",
    }

    if model_cfg.cache_dir:
        model_kwargs["cache_dir"] = model_cfg.cache_dir
    if model_cfg.use_flash_attention:
        model_kwargs["attn_implementation"] = "flash_attention_2"

    return AutoModelForCausalLM.from_pretrained(**model_kwargs)

In [ ]:
@dataclass
class DataConfi:
    dataset_name: str | None = None
    dataset_config: str | None = None
    train_file: str | None = None
    val_file: str | None = None
    text_column: str = "text"
    prompt_column: str | None = "instruction"
    response_column: str | None = "output"
    max_seq_length: int = 2048
    instruction_template: str = "### Instruction:\n"
    response_template: str = "### Response:\n"
    val_split: float = 0.05
    num_proc: int = 4

Todo:
- download dataset
- process and prepare dataset
- save dataset
- tokenize dataset

In [ ]:
def download_and_process_dataset():
    pass

In [ ]:
# from datasets import load_dataset
from datasets import load_dataset

ds = load_dataset("yahma/alpaca-cleaned")
# save to /home/emmanuel/Desktop/research_peft/datasets
ds.save_to_disk("/home/emmanuel/Desktop/research_peft/dataset/alpaca-cleaned")

ImportError: cannot import name 'load_dataset' from 'datasets' (/home/emmanuel/Desktop/research_peft/datasets/__init__.py)

In [ ]:
import pandas as pd

df = pd.read_json("hf://datasets/yahma/alpaca-cleaned/alpaca_data_cleaned.json")
# save to /home/emmanuel/Desktop/research_peft/datasets

df.to_json(
    "/home/emmanuel/Desktop/research_peft/dataset/alpaca_data_cleaned.json",
    orient="records",
    indent=2,
)

In [15]:
df.head()

,instruction,input,output
0,Give three tips for staying healthy.,,1. Eat a balanced and nutritious diet: Make su...
1,What are the three primary colors?,,"The three primary colors are red, blue, and ye..."
2,Describe the structure of an atom.,,An atom is the basic building block of all mat...
3,How can we reduce air pollution?,,There are several ways to reduce air pollution...
4,Pretend you are a project manager of a constru...,,I had to make a difficult decision when I was ...


In [4]:
from scripts.config import DataConfig
from scripts.trainer import format_instruction, load_and_prepare_data

data = "dataset/alpaca_data_cleaned.json"

data_config = DataConfig(train_file=data)
train_data, val_data = load_and_prepare_data(data_config)

Generating train split: 51760 examples [00:06, 7527.32 examples/s]
INFO:scripts.trainer:Train: 49,172 | Val: 2,588


In [13]:
train_data.get("instruction")

AttributeError: 'Dataset' object has no attribute 'get'

In [16]:
for item in train_data:
    print(item.get("instruction"))
    print(item.get("input"))
    print(item.get("outputs"))
    break

Find a book title that uses this phrase: "A Tale of Two Cities".

None


In [17]:
train_data

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 49172
})

In [ ]:
# Dataset Preparation
def formt_instruction(sample: dict, data_cfg: DataConfig) -> str:
    """Format dataset into instruction-following format."""
    instruct = []
    for item in sample:
        instruction = item.get(data_cfg.prompt_column, "")
        context = item.get("input", "")
        response = item.get(data_cfg.response_column, "")

        if context:
            instruct.append(
                f"{data_cfg.instruction_template}{instruction}\n\n"
                f"### Context:\n{context}\n\n"
                f"{data_cfg.response_template}{response}"
            )
        instruct.append(
            f"{data_cfg.instruction_template}{instruction}\n\n{data_cfg.response_template}{response}"
        )

    # ds = Dataset.from_list(instruct)
    return instruct

In [26]:
processed_train_data = format_instruction(train_data, data_config)
processed_val_data = format_instruction(val_data, data_config)

In [28]:
processed_train_data[:10]

['### Instruction:\nFind a book title that uses this phrase: "A Tale of Two Cities".\n\n### Response:\nA title of a book that uses the phrase "A Tale of Two Cities" is "A Tale of Two Cities" by Charles Dickens.',
 '### Instruction:\nConstruct a proverb that encapsulates the concept of freedom.\n\n### Response:\n"Freedom is not given, it is won through determination and courage."',
 "### Instruction:\nDescribe the effect of the Great Plague of 1665 on England.\n\n### Response:\nThe Great Plague of 1665 was a major event in the history of England, which greatly affected the country both socially and economically. The outbreak, which is believed to have originated in the parish of St. Giles-in-the-Fields, outside of London, spread rapidly throughout the city and eventually reached other areas of the country. It is estimated that between 75,000 and 100,000 people, which was 15% to 20% of London's population, died from the disease.\n\nOne of the most significant impacts the Great Plague had

In [40]:
for i in val_data:
    print(i)
    break

{'instruction': 'Find the linked countries.', 'input': 'Mount Everest is the highest peak in the world and it is located in Nepal.', 'output': 'Nepal is the only country mentioned in the given input sentence. Mount Everest, the highest peak in the world, is located in Nepal, and it is also bordered by Tibet, an autonomous region of China.'}


In [43]:
# from dataset.download import ALPACA_PROMPT
ALPACA_PROMPT = (
    # "Below is an instruction that describes a task{context_str}. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "{input_block}"
    "### Response:\n{output}"
)

for i in val_data:
    [ALPACA_PROMPT.format((row) for row in i)]
    break

KeyError: 'instruction'

In [ ]:
import re
import string


def normalize_answer(text):
    """Standard normalization for QA evaluation."""
    text = text.lower()
    text = re.sub(r"^(a|an|the)\s+", "", text)  # Remove leading articles
    text = "".join(char for char in text if char not in set(string.punctuation))

    return " ".join(text.split())  # Normalize whitespace


def compute_exact_match_score(prediction, references):
    """Returns 1.0 if normalized prediction matches any reference, else 0.0."""
    normalized_pred = normalize_answer(prediction)
    normalized_refs = [normalize_answer(ref) for ref in references]
    return float(normalized_pred in normalized_refs)